# Fintech Next Best Action — LinUCB Contextual Bandit

Personalized action selection on synthetic multi-product fintech customers.

Run all cells to see learning curves and action distributions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

ACTIONS = [
    'savings_boost', 'premium_account', 'credit_card_upsell',
    'personal_loan', 'investment_product', 'retention_nudge',
    'cashback_offer', 'do_nothing'
]

In [ ]:
def generate_synthetic_customers(n=2000):
    return pd.DataFrame({
        'customer_id': range(1, n + 1),
        'age': np.random.randint(22, 65, n),
        'tenure_months': np.random.exponential(18, n).astype(int) + 1,
        'balance': np.random.lognormal(8.5, 1.2, n).round(2),
        'monthly_tx': np.random.poisson(12, n),
        'products_held': np.random.randint(1, 5, n),
        'engagement_score': np.random.beta(2, 5, n),
        'churn_risk': np.random.beta(1.5, 4, n),
        'income_proxy': np.random.lognormal(10.2, 0.6, n).round(0),
    })

def context_vector(row):
    return np.array([
        row['age'] / 65,
        min(row['tenure_months'] / 60, 1),
        np.log1p(row['balance']) / 12,
        min(row['monthly_tx'] / 30, 1),
        row['products_held'] / 5,
        row['engagement_score'],
        row['churn_risk'],
        np.log1p(row['income_proxy']) / 12,
        1.0
    ])

class LinUCB:
    def __init__(self, n_actions, n_features, alpha=1.0):
        self.n_actions = n_actions
        self.n_features = n_features
        self.alpha = alpha
        self.A = [np.eye(n_features) for _ in range(n_actions)]
        self.b = [np.zeros(n_features) for _ in range(n_actions)]

    def select_action(self, x):
        p = []
        for a in range(self.n_actions):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]
            conf = self.alpha * np.sqrt(x @ A_inv @ x)
            p.append(theta @ x + conf)
        return int(np.argmax(p))

    def update(self, action, x, reward):
        self.A[action] += np.outer(x, x)
        self.b[action] += reward * x

def simulate_reward(customer, action_idx):
    a = ACTIONS[action_idx]
    base = 0.0
    if a == 'savings_boost' and customer['balance'] > 5000 and customer['engagement_score'] > 0.4:
        base = 0.35
    elif a == 'premium_account' and customer['income_proxy'] > 40000 and customer['products_held'] >= 2:
        base = 0.28
    elif a == 'credit_card_upsell' and customer['monthly_tx'] > 8 and customer['age'] < 45:
        base = 0.25
    elif a == 'personal_loan' and customer['churn_risk'] < 0.3 and customer['tenure_months'] > 12:
        base = 0.22
    elif a == 'investment_product' and customer['balance'] > 15000:
        base = 0.30
    elif a == 'retention_nudge' and customer['churn_risk'] > 0.5:
        base = 0.40
    elif a == 'cashback_offer' and customer['monthly_tx'] > 15:
        base = 0.18
    elif a == 'do_nothing':
        base = 0.05
    noise = np.random.normal(0, 0.08)
    return float(np.clip(base + noise, 0, 1))

customers = generate_synthetic_customers(2000)
print(f'Customers generated: {len(customers)}')
customers.head()

In [ ]:
n_features = 9
bandit = LinUCB(len(ACTIONS), n_features, alpha=0.8)

cumulative = []
action_history = []
reward_history = []
cum = 0.0

for _, row in customers.iterrows():
    x = context_vector(row)
    a = bandit.select_action(x)
    r = simulate_reward(row, a)
    bandit.update(a, x, r)
    cum += r
    cumulative.append(cum)
    action_history.append(a)
    reward_history.append(r)

print(f'Final cumulative reward: {cum:.2f}')
print(f'Average reward: {np.mean(reward_history):.4f}')

## Learning Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(cumulative, color='#2E86AB', lw=1.5)
plt.title('Cumulative Reward — LinUCB Contextual Bandit')
plt.xlabel('Customers processed')
plt.ylabel('Cumulative reward')
plt.tight_layout()
plt.show()

## Action Distribution Over Time

In [ ]:
window = 200
action_df = pd.DataFrame({'action': action_history})
action_df['window'] = (action_df.index // window)
counts = action_df.groupby(['window', 'action']).size().unstack(fill_value=0)
counts.columns = [ACTIONS[i] for i in counts.columns]

counts.plot(kind='area', stacked=True, figsize=(11, 5), colormap='tab10', alpha=0.85)
plt.title('Action Selection Over Time (windows of 200 customers)')
plt.xlabel('Time window')
plt.ylabel('Selections')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Recommended next steps
1. Replace synthetic features with real warehouse features (balance, product holdings, recent activity).
2. Define the reward as incremental contribution margin or retention lift from a holdout.
3. Deploy as a batch scoring job into the CRM / marketing automation platform.
4. Keep a controlled exploration rate and monitor regret.